In [1]:
from datetime import timedelta
import datetime
import io
from PIL import Image
from cairosvg import svg2png
import numpy as np
from glider import GoProVideoMetadata, IGCFile
import cv2
from matplotlib import pyplot as plt
from zoneinfo import ZoneInfo

In [2]:
video_filename = 'GX010108.MP4'
# out_filename = 'GX020107_enriched.mp4'
out_filename_no_audio = 'noaudio.mp4'
igc_data = IGCFile("2026-05-01-XTR-A68A17562166-01.IGC")
# TODO: check start time of video when cut by gopro (2xx)
# meta0 = GoProVideoMetadata.from_video("GX010107.MP4")
gopro_meta = GoProVideoMetadata.from_video(video_filename)
gopro_meta.creation_time -= timedelta(hours=2)

In [3]:

gopro_meta = GoProVideoMetadata.from_video(video_filename)

In [4]:
# Compass Data
# compass_bg = 
# svg_data = open("compass-north-svgrepo-com.svg", "rb").read()
# png_data = svg2png(bytestring=svg_data, output_width=100)

compass_position = (10,10)
compass_background = Image.open("compass_background.png").convert("RGBA").resize((200, 200), Image.Resampling.LANCZOS)
needle = Image.open("compass_needle.png").convert("RGBA").resize((200, 200), Image.Resampling.LANCZOS)
needle_center = [x/2 for x in needle.size]

In [ ]:
def draw_hud_with_box(current_alt, frame):
    hud_green = '#00FF41'
    top_bot = 25
    major_ticks = 10
    minor_thicks = 2
    fig, ax = plt.subplots(figsize=(4, 8))
    # ax.set_facecolor('black')
    
    # HUD Tape Line (Right Spine)
    # for spine in ['top', 'bottom', 'left']:
    #     ax.spines[spine].set_visible(False)
    # ax.spines['right'].set_color(hud_green)
    # ax.spines['right'].set_linewidth(2)
    ax.spines.clear()

    # Set view range (e.g., show 300 units above/below current)
    ax.set_ylim(current_alt - top_bot, current_alt + top_bot)

    # Draw ticks and labels
    start_tick = int((current_alt - top_bot) // minor_thicks * minor_thicks)
    for val in range(start_tick, int(current_alt + top_bot * 1.25), minor_thicks):
        is_major = val % major_ticks == 0
        tick_len = 0.15 if is_major else 0.08
        
        # Draw tick lines (at x=0.7 to leave room for the box on the right)
        ax.plot([0.7 - tick_len, 0.7], [val, val], color=hud_green, 
                lw=2 if is_major else 1, transform=ax.get_yaxis_transform())
        
        if is_major:
            ax.text(0.5, val, f"{val:04d}", color=hud_green, fontsize=20, 
                    ha='right', va='center', family='monospace', transform=ax.get_yaxis_transform())

    # THE BOX: Current Altitude Readout
    box_style = dict(boxstyle='square,pad=0.3', facecolor='none', edgecolor=hud_green, linewidth=2)
    ax.text(0.82, current_alt, f"{round(current_alt):05d}", color=hud_green, fontsize=20,
            ha='left', va='center', family='monospace', fontweight='bold',
            bbox=box_style, transform=ax.get_yaxis_transform())
    
    # Connecting line from tape to box
    ax.plot([0.7, 0.82], [current_alt, current_alt], color=hud_green, lw=2, transform=ax.get_yaxis_transform())

    ax.set_xticks([])
    ax.set_yticks([])
    # plt.savefig(output_path, facecolor='black', bbox_inches='tight')
    import io
    buf = io.BytesIO()
    fig.savefig(buf, format='png', transparent=True)
    plt.close(fig)
    buf.seek(0)
    img = Image.open(buf).convert("RGBA")
    img = img.crop([120, 0, img.size[0], img.size[1]])
    bg = Image.fromarray(frame)
    pos_y = round(bg.size[1] / 2 - img.size[1] / 2)
    bg.paste(img, ( 0, pos_y), img)
    return np.array(bg)

# img = draw_hud_with_box(1200, frame)
# Image.fromarray(img)


def draw_fa18_high_res_compass(heading, frame):
    hud_green = '#00FF41'
    current_hdg = heading % 360
    
    fig, ax = plt.subplots(figsize=(10, 4), facecolor='black')
    # ax.set_facecolor('black')
    
    radius, center_y, view_span = 10.0, -9.2, 20  # span reduced slightly for clarity
    
    # Draw curved rail
    arc_ans = np.linspace(np.radians(90 - view_span), np.radians(90 + view_span), 100)
    ax.plot(radius * np.cos(arc_ans), radius * np.sin(arc_ans) + center_y, color=hud_green, lw=2)

    # 1-degree resolution loop
    search_min = int(np.floor(current_hdg - view_span))
    search_max = int(np.ceil(current_hdg + view_span))

    for h in range(search_min, search_max + 1):
        offset = h - current_hdg
        
        if abs(offset) <= view_span:
            angle_rad = np.radians(90 - offset)
            display_deg = h % 360
            
            # Determine tick style
            if display_deg % 10 == 0:
                r_in = 9.7  # Long
                lw = 2
            elif display_deg % 5 == 0:
                r_in = 9.82 # Medium
                lw = 1.5
            else:
                r_in = 9.92 # Short (1-degree resolution)
                lw = 0.8
                
            # Plot tick
            ax.plot([r_in * np.cos(angle_rad), radius * np.cos(angle_rad)],
                    [r_in * np.sin(angle_rad) + center_y, radius * np.sin(angle_rad) + center_y],
                    color=hud_green, lw=lw)
            
            # Labels for 10-degree increments
            if display_deg % 10 == 0:
                label = {0:'N', 90:'E', 180:'S', 270:'W'}.get(display_deg, f"{int(display_deg/10):02d}")
                tx, ty = 9.3 * np.cos(angle_rad), 9.3 * np.sin(angle_rad) + center_y
                ax.text(tx, ty, label, color=hud_green, fontsize=20, 
                        ha='center', va='center', family='monospace', fontweight='bold')

    # HUD center-markers
    ax.text(0, 0.9, '∨', color=hud_green, fontsize=24, ha='center', fontweight='bold')
    ax.text(0, 1.25, f"{int(current_hdg):03d}", color=hud_green, fontsize=18, 
            ha='center', family='monospace', fontweight='bold',
            bbox=dict(boxstyle='square,pad=0.2', fc='none', ec=hud_green, lw=2))

    ax.set_xlim(-3.5, 3.5)
    ax.set_ylim(-0.5, 1.8)
    ax.set_aspect('equal')
    ax.axis('off')

    import io
    buf = io.BytesIO()
    fig.savefig(buf, format='png', transparent=True)
    plt.close(fig)
    buf.seek(0)
    img = Image.open(buf).convert("RGBA")
    img = img.crop([0, 100, 1000, 400])
    bg = Image.fromarray(frame)
    pos_x = round(bg.size[0] / 2 - img.size[0] / 2)
    bg.paste(img, (pos_x, 0), img)
    return np.array(bg)



def draw_compass(frame: np.array, heading: float):
    img = Image.fromarray(frame)
    rotated_overlay = needle.rotate(-heading, center=needle_center, resample=Image.BICUBIC)
    img.paste(compass_background, compass_position, compass_background)
    img.paste(rotated_overlay, compass_position, rotated_overlay)
    frame = np.array(img)
    return frame

def draw_altitude_line(igc: IGCFile, time: datetime.datetime, frame):
    backwards, forwards = igc.get_altitude_progression(time)
    y = [*-backwards, *-forwards] # Invert the values (image pixel direction)
    if not y:
        return frame
    y += min(y) # Make values positive
    x = [x for x in range(len(y))]
    x_image_size = 280
    y_image_size = 100
    plot_position = (240, 90)
    x_scaled = plot_position[0] + (x - np.min(x)) / (np.max(x) - np.min(x)) * x_image_size
    y_scaled = plot_position[1] + (y - np.min(y)) / (np.max(y) - np.min(y)) * y_image_size

    x = x_scaled
    y = y_scaled

    points = np.column_stack((x, y)).astype(np.int32)

    # 3. Draw the Fading Shadow (Negative Y direction)
    # In image coordinates, "down" is the positive Y-axis.
    shadow_layers = 25
    max_alpha = 0.2

    for i in range(shadow_layers, 0, -1):
        # Create a transparent overlay for the current shadow layer
        overlay = frame.copy()
        
        # Calculate transparency: fades out as it moves away from the line
        alpha = max_alpha * (1 - i / shadow_layers)**1.5
        
        # Offset the points downward to create the shadow effect
        offset_points = points.copy()
        offset_points[:, 1] += i
        
        # Draw the shadow line on the overlay
        cv2.polylines(overlay, [offset_points], isClosed=False, color=(75, 150, 200), thickness=5)
        
        # Blend the overlay back into the main image
        frame = cv2.addWeighted(overlay, alpha, frame, 1 - alpha, 0)

    # 4. Draw the main curve on top
    cv2.polylines(frame, [points[0:int(len(backwards))]], isClosed=False, color=(0, 0, 0), thickness=4, lineType=cv2.LINE_AA)
    cv2.polylines(frame, [points[int(len(backwards)):]], isClosed=False, color=(0, 255, 0), thickness=4, lineType=cv2.LINE_AA)
    return frame

In [6]:
start_datetime = gopro_meta.creation_time - timedelta(hours=2)
cap = cv2.VideoCapture(video_filename)

fps = cap.get(cv2.CAP_PROP_FPS)
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(out_filename_no_audio, fourcc, fps, (width, height))
frame_count = 0

In [7]:
current_frame_time = start_datetime + timedelta(seconds=frame_count/fps)
heading_value = igc_data.get_bearing_at_time(current_frame_time).bearing_deg
heading_value, current_frame_time

(np.float64(110.0142021450566),
 datetime.datetime(2026, 5, 1, 10, 14, 13, tzinfo=datetime.timezone.utc))

In [9]:

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
    current_frame_time = start_datetime + timedelta(seconds=frame_count/fps)
    heading_value = igc_data.get_bearing_at_time(current_frame_time).bearing_deg
    altitude = igc_data.get_gps_altitude_at_time(current_frame_time)
    print(altitude)

    frame = cv2.putText(
        frame,
        current_frame_time.replace(microsecond=0).astimezone(ZoneInfo("Europe/Zurich")).isoformat(),
        (25, frame.shape[0]-25), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 0, 0), 3)
    frame = draw_hud_with_box(altitude, frame)
    # frame = draw_fa18_high_res_compass(heading_value, frame)
    # frame = draw_compass(frame, heading_value)
    # frame = draw_altitude_line(igc_data, current_frame_time, frame)
    out.write(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    frame_count += 1
    if frame_count > 1000:
        break

cap.release()
out.release()
print("Export Complete!")

1100.0


ValueError: Unknown format code 'd' for object of type 'float'

Error in callback <function _draw_all_if_interactive at 0x000001D5E70F7880> (for post_execute), with arguments args (),kwargs {}:


AttributeError: 'Spines' object does not contain a 'bottom' spine

AttributeError: 'Spines' object does not contain a 'bottom' spine

<Figure size 400x800 with 1 Axes>

In [ ]:
# ffmpeg -i noaudio.mp4 -i GX020107.MP4 -c:v "copy" -c:a "aac" -map "0:v:0" -map "1:a:0" -shortest test.mp4